# Retrieval-Augmented Generation (RAG)

**RAG** combines information retrieval with text generation. Instead of relying only on what a language model learned during training, a RAG system first finds relevant information from a knowledge source and then gives that information to the model as context for its answer.

## How this notebook's RAG pipeline works

1. **Load and prepare source data:** fetch a web page, clean its text, and split it into smaller chunks.
2. **Index the chunks:** an embedding model converts each chunk into a vector, and Chroma stores the vectors alongside the original text.
3. **Retrieve relevant information:** embed a question and search Chroma for chunks with similar meaning.
4. **Generate an answer:** combine the retrieved chunks into a context string, include it with the question in a prompt, and ask the language model to answer from that context.

In short: **question → retrieve relevant chunks → provide them as context → generate an answer**. The embedding vectors help find relevant text; the language model receives the retrieved text, not the vectors themselves.

## 1. Load source documents

Fetch the Google blog page and load its content as LangChain `Document` objects. These are the source documents that the RAG pipeline will clean, split, embed, store, and retrieve.

In [1]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(web_path='https://blog.google/intl/en-in/products/introducing-gemini-4-argon/')

docs = loader.load()

C:\Users\Rayyan\AppData\Local\Temp\ipykernel_4472\1359690293.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


## 2. Split documents into chunks

Configure a recursive splitter to break long documents into chunks of about 1,000 characters, with 200 characters of overlap. Overlap helps preserve context across chunk boundaries. This first split is an initial pass; the next cell cleans the source and splits it again.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
        is_separator_regex=False,
    )
split_texts = text_splitter.split_documents(docs)

## 3. Clean and re-split the source text

Remove stray slash characters and collapse repeated whitespace in each document. Then split the cleaned documents again so the chunks used by the rest of the pipeline contain the cleaned text.

In [3]:
import re

def clean_scraped_text(text: str) -> str:
    # 1. Remove unwanted standalone slashes or stray marks (like /and or / )
    text = re.sub(r'\s+/\s*', ' ', text) 
    
    # 2. Collapse multiple spaces/newlines into clean spacing
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

# Apply cleaning to all documents before splitting
for doc in docs:
    doc.page_content = clean_scraped_text(doc.page_content)

# Now split your cleaned documents safely
split_texts = text_splitter.split_documents(docs)


## 4. Count the chunks

Display how many text chunks were created. This is a quick check that splitting produced data before you inspect or store it.

In [4]:
len(split_texts)

27

## 5. Inspect the chunk contents

Print each chunk with a numbered label and separator. Reviewing these examples helps you check whether the source was cleaned and split into useful pieces.

In [5]:
for i in range(len(split_texts)):
    print(f"Chunk {i+1}:")
    print(split_texts[i].page_content)
    print("\n---\n")

Chunk 1:
Gemini 4 Argon is now available for everyone Skip to main content India Blog Gemini 4 Argon: Built to solve your hardest problems Latest stories Product updates Company news Feed Latest stories Product updates Company news Feed Gemini 4 Argon: Built to solve your hardest problems Share x.com Facebook Linkedin Mail Copy link [] Preferences Global (English) Africa (English) Australia (English) Brasil (Português) Canada (English) Canada (Français) Česko (Čeština) Deutschland (Deutsch) España (Español) France (Français) Greece (Ελληνικά) India (English) Indonesia (Bahasa Indonesia) Ireland (English) Italia (Italiano) 日本 (日本語) 대한민국 (한국어) Latinoamérica (Español) Malaysia (English) الشرق الأوسط وشمال أفريقيا (اللغة العربية) MENA (English) Nederlands (Nederland) New Zealand (English) Polska (Polski) Portugal (Português) România (Română) Sverige (Svenska) ประเทศไทย (ไทย) Türkiye (Türkçe) 台灣 (中文) Links Press corner RSS feed x.com Facebook Linkedin Mail Copy link Breadcrumb Home Product 

## 6. Set up the embedding model

Load `all-mpnet-base-v2`, which converts text into numeric vectors for semantic search. The optional Hugging Face token can raise download limits; the model is configured to run on the CPU here (change `device` to `cuda` if available).

In [6]:
from getpass import getpass
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "sentence-transformers/all-mpnet-base-v2"
hf_token = getpass("Hugging Face token (leave blank for public models): ").strip()
model_kwargs = {"device": "cpu"}  # Use "cuda" if you have a GPU
if hf_token:
    model_kwargs["token"] = hf_token
encode_kwargs = {"normalize_embeddings": False}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 7. Set up the Chroma vector store

Create or open the `my_collection` collection in Chroma, persist it in `./chroma_persist`, and connect the embedding model. Chroma will use that model to embed text when documents are added and queries are searched.

In [7]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name="my_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_persist",

)

## 8. Add the chunks to Chroma

Store the cleaned chunks in the vector store. Chroma creates embeddings for the text and saves the document content and associated data; the returned IDs identify the stored records.

In [8]:
#Adding the documents to the vectorstore
vectorstore.add_documents(split_texts)

['78885deb-12f3-471d-8905-5ad9a8edeb14',
 'c2fc351d-093f-439a-a9c1-5e38cc5a70fa',
 '477a66cd-276a-4e31-acab-fb3382c3ffc4',
 '3a5271ea-8dbb-4e6c-b08c-0791bc86ae58',
 'ad7b5d3a-5f0e-42ea-886b-756d2751051b',
 'ee0427d2-7d61-4bb1-88d5-5d361a27e802',
 '3ea561cf-2722-475a-962f-2e84434f7963',
 '781dcae5-43cb-4b1a-959d-fb2676978798',
 '117b2564-65bc-4292-90d1-195f4fac2a08',
 'a42fb842-83e6-40e9-bf2d-509bfe938494',
 'cc9073ae-4fe0-4faf-8b05-5d2bbaa4b69f',
 '1889ac84-18ab-459a-8e23-831339d5ba6c',
 '6ac44cee-2290-484a-8df7-db3e148fb2df',
 'b678116e-5950-446f-b51f-df798d9e8c61',
 '033a0c9a-8432-4ca1-a370-d53a872f43f0',
 'b87896bd-60ff-4211-b8cc-0bd37c44f041',
 '46624f31-da48-44a7-a60f-99ce59550319',
 'd48698a1-c31e-4a45-8653-84dcdf67c33b',
 '13709ff5-060b-48ef-95aa-9c0a63b30ee0',
 'e599783d-bade-4218-8f13-878ba3ca46da',
 'ff98ccf8-d0ee-403d-be24-29ee31b5fa88',
 '0df0150a-c211-4daa-aaff-1d4812425740',
 '2dd3adf7-6c1e-4dab-ab62-495219a61001',
 '1ab39dc7-b208-427f-91a7-b9a0e382e052',
 'f7ee4a9a-f377-

## 9. Inspect the stored collection

Fetch and print the collection's stored records, then print the number of records. This is a basic check that the chunks were added to Chroma.

In [9]:
# Get all stored records (IDs, documents, and metadata)
data = vectorstore._collection.get()
print(data)

# Count total documents in the collection
print(vectorstore._collection.count())


{'ids': ['563458d8-5f27-4786-9a35-bd0e96bd06c5', '61253dac-de21-4bb0-b076-3750c09619bb', '66f559f1-8d07-4a07-9f2f-82583862504d', '8ce98f2d-5877-464c-8cb6-2e676df568fe', '0f3de849-f8ea-4a6c-b8d3-bebd691d962e', '9b0c25af-25fe-4b05-96e4-2987cf3d8582', '4d5d9f5b-96c7-42c4-befc-979513db1a57', 'a413c394-a9b8-411c-906a-85cb7237734f', 'afd14fe5-9889-43ad-85b3-6e99b1110f5f', '0764ef1e-ed6c-4448-95d4-fddc60e86a4c', '1181666a-071c-442f-a6dc-d4c631ab5db2', 'e3b9c481-e2a5-4fd6-8302-bd192927ee72', '57c58e70-1105-4b3b-a9fb-eba5b108a535', '97cbd44b-9990-4df5-82ec-43cd5e43feab', 'becbfe7f-180d-4b9d-b040-a4e50697647f', 'f8e0fbe9-7778-4472-a2a0-e1742411c705', '478dfe46-8a9e-42a7-b997-beeaf0d6774a', 'cf43e60f-1ef5-4cc7-b5bd-9e6a420a692c', '17c3294e-8f9e-4dc0-afad-6efabee4bad6', '7168ad08-2044-4a5e-aaa3-a9ec1216e155', '20fa6659-408e-4eb5-ae2e-5b66cccd96f2', 'd4fc79ca-7779-4e0f-a0ec-cfd141fd175d', '18d9ae8e-f594-421d-b607-10f6fa792237', '89bb0aa1-31b5-4b85-9f55-2a7bae50252c', '16911337-bb0b-420f-b868-8d4da1

## 10. Optional: inspect one stored record

The commented example shows how to fetch a specific Chroma record by ID, including its embedding and document text. Uncomment it and use a valid ID if you want to inspect one item in more detail.

In [10]:
# print("\n Collection 1 - ", vectorstore._collection.get(ids=['563458d8-5f27-4786-9a35-bd0e96bd06c5'], include=['embeddings', 'documents']))

## 11. Try a similarity search

Search Chroma for the three chunks most similar to the example query, `Gemini 4 Argon`, and print the returned documents. The query is embedded with the same embedding model before matching.

In [11]:
# Augumentation of the documents in the vectorstore

retrieved_docs = vectorstore.similarity_search("Gemini 4 Argon", k=3)
print("Retrieved Documents:", retrieved_docs)

Retrieved Documents: [Document(id='16911337-bb0b-420f-b868-8d4da14bff52', metadata={'title': 'Gemini 4 Argon is now available for everyone', 'source': 'https://blog.google/intl/en-in/products/introducing-gemini-4-argon/', 'description': 'We’re giving developers, enterprises, scientists, and consumers access to Gemini 4 Argon, our latest frontier model.', 'language': 'en-in'}, page_content="enterprises: Try it in Gemini for business, the API, and in Google Antigravity.For everyday users: Try it starting today in the Gemini app and AI Mode in Search if you’re a Google AI Ultra or Pro subscriber.For scientists & researchers: Apply for a chance to get access to Google Antigravity’s highest usage tier free of charge for 6 months (onboarded on a rolling basis), subject to eligibility, capacity, and selection. Applications are now open.For cybersecurity defenders: Apply to join the Fairwind Program to get access to Argon without cyber guardrails on a rolling basis.Gemini 4 Argon is just the b

## 12. Configure a reusable retriever

Wrap the vector store as a retriever configured to return three results. A retriever provides a standard interface for fetching relevant documents; the final cell below performs its search directly on `vectorstore`.

In [12]:
retreiver = vectorstore.as_retriever(search_kwargs={"k": 3})

## 13. Set up the chat model

Configure Ollama to run the local `llama3.1:8b` model. The temperature and other options control generation behavior; Ollama must be installed and the model available locally.

In [13]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1:8b", temperature=0.7, max_tokens=512, stop=["\n\n"], top_p=0.9)

## 14. Retrieve context and generate an answer

Build a prompt that asks the model to answer only from the retrieved context. For the question, retrieve the four most similar documents, join their `page_content` into one context string, and pass both values through the prompt, chat model, and string output parser.

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "Answer using only the context below. If the answer isn't there, say you don't know.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
)

rag_chain = prompt | llm | StrOutputParser()

question = "Tell me about Gemini 4 Argon"
docs = vectorstore.similarity_search(question, k=4)
context = "\n\n".join(doc.page_content for doc in docs)

answer = rag_chain.invoke({
    "context": context,
    "question": question,
})

print(answer)

Gemini 4 Argon is now available for everyone. It's built to solve the hardest problems and is available in various languages, including English, Spanish, French, and more. 
